In [1]:
import pandas as pd
import numpy as np

In [2]:
file_path = "data/wemabank.json"
df = pd.read_json(file_path)

df.head()

,date,open,high,low,close,volume
0,2016-05-25,NaN,NaN,NaN,0.78,NaN
1,2016-05-26,NaN,NaN,NaN,0.78,NaN
2,2016-05-27,NaN,NaN,NaN,0.80,NaN
3,2016-05-31,NaN,NaN,NaN,0.78,NaN
4,2016-06-01,NaN,NaN,NaN,0.76,NaN


In [40]:
from pathlib import Path
import pandas as pd


def load_all_stock_data(folder_path: str) -> pd.DataFrame:
    """
    Load and combine all JSON stock files in a folder.

    Parameters
    ----------
    folder_path : str
        Path to the folder containing JSON files.

    Returns
    -------
    pd.DataFrame
        Combined dataframe with an additional 'stock_name' column.
    """
    folder = Path(folder_path)

    dfs = []

    for file in folder.glob("*.json"):
        try:
            df = pd.read_json(file)
            df["stock_name"] = file.stem  # Filename without .json
            dfs.append(df)
        except Exception as e:
            print(f"Skipping {file.name}: {e}")

    if not dfs:
        return pd.DataFrame()

    combined_df = pd.concat(dfs, ignore_index=True)

    return combined_df

In [41]:
df =  load_all_stock_data("data")
df.shape

Skipping _metadata.json: If using all scalar values, you must pass an index


(306872, 7)

In [42]:
df.head()

,date,open,high,low,close,volume,stock_name
0,2016-05-31,NaN,NaN,NaN,1.25,NaN,abbeybds
1,2016-06-06,NaN,NaN,NaN,1.25,NaN,abbeybds
2,2016-06-20,NaN,NaN,NaN,1.25,NaN,abbeybds
3,2016-06-21,NaN,NaN,NaN,1.25,NaN,abbeybds
4,2016-06-23,NaN,NaN,NaN,1.25,NaN,abbeybds


In [44]:
df.columns

Index(['date', 'open', 'high', 'low', 'close', 'volume', 'stock_name'], dtype='object')

In [43]:
df.to_csv("data.csv")

In [6]:
df.stock_name.nunique()

146

In [7]:
df.tail()

,date,open,high,low,close,volume,stock_name
306867,2026-06-22,NaN,NaN,NaN,23.40,NaN,zichis
306868,2026-06-23,NaN,NaN,NaN,24.67,NaN,zichis
306869,2026-06-24,NaN,NaN,NaN,24.90,NaN,zichis
306870,2026-06-25,NaN,NaN,NaN,24.90,NaN,zichis
306871,2026-06-26,NaN,NaN,NaN,24.95,NaN,zichis


In [8]:
import pandas as pd

# Keep only useful columns
df = df[['date', 'stock_name', 'close']].copy()

df['date'] = pd.to_datetime(df['date'])

df['stock_name'] = df['stock_name'].str.upper()

df = df.sort_values(['stock_name', 'date']).reset_index(drop=True)

df.head()

,date,stock_name,close
0,2016-05-31,ABBEYBDS,1.25
1,2016-06-06,ABBEYBDS,1.25
2,2016-06-20,ABBEYBDS,1.25
3,2016-06-21,ABBEYBDS,1.25
4,2016-06-23,ABBEYBDS,1.25


In [9]:
def engineer_features(group):
    group = group.sort_values("date").copy()

    # Daily return
    group["return"] = group["close"].pct_change()

    # Log return
    group["log_return"] = np.log(group["close"] / group["close"].shift(1))

    # Moving averages
    group["ma5"] = group["close"].rolling(5).mean()
    group["ma10"] = group["close"].rolling(10).mean()
    group["ma20"] = group["close"].rolling(20).mean()
    group["ma50"] = group["close"].rolling(50).mean()

    # Exponential moving averages
    group["ema10"] = group["close"].ewm(span=10).mean()
    group["ema20"] = group["close"].ewm(span=20).mean()

    # Volatility
    group["volatility10"] = group["return"].rolling(10).std()

    # Momentum
    group["momentum10"] = group["close"] - group["close"].shift(10)

    # Rate of Change
    group["roc10"] = group["close"].pct_change(10)

    # Rolling max/min
    group["rolling_max20"] = group["close"].rolling(20).max()
    group["rolling_min20"] = group["close"].rolling(20).min()

    # Drawdown
    rolling_peak = group["close"].cummax()

    group["drawdown"] = (
        group["close"] - rolling_peak
    ) / rolling_peak

    return group

In [10]:
df = (
    df.groupby("stock_name", group_keys=False)
      .apply(engineer_features)
      .reset_index(drop=True)
)

C:\Users\Tunji.obidokun\AppData\Local\Temp\ipykernel_33104\2448234370.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(engineer_features)


In [11]:
def calculate_rsi(close, period=14):

    delta = close.diff()

    gain = delta.clip(lower=0)

    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(period).mean()

    avg_loss = loss.rolling(period).mean()

    rs = avg_gain / avg_loss

    rsi = 100 - (100 / (1 + rs))

    return rsi

In [12]:
df["rsi"] = (
    df.groupby("stock_name")["close"]
      .transform(calculate_rsi)
)

In [13]:
rolling_mean = (
    df.groupby("stock_name")["close"]
      .transform(lambda x: x.rolling(20).mean())
)

rolling_std = (
    df.groupby("stock_name")["close"]
      .transform(lambda x: x.rolling(20).std())
)

df["bb_upper"] = rolling_mean + 2 * rolling_std

df["bb_lower"] = rolling_mean - 2 * rolling_std

In [14]:
def macd(close):

    ema12 = close.ewm(span=12).mean()

    ema26 = close.ewm(span=26).mean()

    macd = ema12 - ema26

    signal = macd.ewm(span=9).mean()

    return macd, signal

In [15]:
df["macd"] = np.nan
df["macd_signal"] = np.nan

for stock, idx in df.groupby("stock_name").groups.items():

    m, s = macd(df.loc[idx, "close"])

    df.loc[idx, "macd"] = m.values
    df.loc[idx, "macd_signal"] = s.values

In [16]:
df["target"] = (
    df.groupby("stock_name")["close"]
      .shift(-1)
)

In [17]:
df["target_next_5_days"] = (
    df.groupby("stock_name")["close"]
      .shift(-5)
)

In [18]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["stock_id"] = le.fit_transform(df["stock_name"])

In [19]:
df = df.dropna().reset_index(drop=True)

In [20]:
df.stock_id.nunique()

141

In [21]:
FEATURES = [
    "stock_id",
    "close",
    "return",
    "log_return",
    "ma5",
    "ma10",
    "ma20",
    "ma50",
    "ema10",
    "ema20",
    "volatility10",
    "momentum10",
    "roc10",
    "rolling_max20",
    "rolling_min20",
    "drawdown",
    "rsi",
    "bb_upper",
    "bb_lower",
    "macd",
    "macd_signal"
]

TARGET = "target"

In [22]:
df.head()

,date,stock_name,close,return,log_return,ma5,ma10,ma20,ma50,ema10,...,rolling_min20,drawdown,rsi,bb_upper,bb_lower,macd,macd_signal,target,target_next_5_days,stock_id
0,2018-07-23,ABBEYBDS,1.17,-0.1,-0.105361,1.274,1.287,1.2935,1.2974,1.276364,...,1.17,-0.1,0.0,1.351638,1.235362,-0.010370,-0.002074,1.17,1.17,0
1,2018-07-24,ABBEYBDS,1.17,0.0,0.000000,1.248,1.274,1.2870,1.2948,1.257025,...,1.17,-0.1,0.0,1.367026,1.206974,-0.018377,-0.005335,1.17,1.17,0
2,2018-07-25,ABBEYBDS,1.17,0.0,0.000000,1.222,1.261,1.2805,1.2922,1.241202,...,1.17,-0.1,0.0,1.375750,1.185250,-0.024441,-0.009156,1.17,1.17,0
3,2018-07-26,ABBEYBDS,1.17,0.0,0.000000,1.196,1.248,1.2740,1.2896,1.228256,...,1.17,-0.1,0.0,1.380702,1.167298,-0.028913,-0.013107,1.17,1.17,0
4,2018-07-27,ABBEYBDS,1.17,0.0,0.000000,1.170,1.235,1.2675,1.2870,1.217664,...,1.17,-0.1,0.0,1.383008,1.151992,-0.032087,-0.016903,1.17,1.17,0


In [23]:
train = df[df["date"] < "2025-01-01"]

test = df[df["date"] >= "2025-01-01"]

X_train = train[FEATURES]
y_train = train[TARGET]

X_test = test[FEATURES]
y_test = test[TARGET]

In [24]:
!pip install xgboost
print("Done")

Done


In [25]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)

model.fit(X_train, y_train)
print("Model trained fully")

Model trained fully


In [26]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
import numpy as np

pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²  : {r2:.4f}")

MAE : 25.1630
RMSE: 270.3822
R²  : 0.7981


In [27]:
X_test.shape

(39595, 21)

In [28]:
# df["target"] = (
#     df.groupby("stock_name")["close"]
#       .pct_change()
#       .shift(-1)
# )

df["signal"] = np.select(
    [
        df["target"] > 0.02,
        df["target"] < -0.02,
    ],
    [
        "BUY",
        "SELL",
    ],
    default="HOLD",
)

In [29]:
df.head()

,date,stock_name,close,return,log_return,ma5,ma10,ma20,ma50,ema10,...,drawdown,rsi,bb_upper,bb_lower,macd,macd_signal,target,target_next_5_days,stock_id,signal
0,2018-07-23,ABBEYBDS,1.17,-0.1,-0.105361,1.274,1.287,1.2935,1.2974,1.276364,...,-0.1,0.0,1.351638,1.235362,-0.010370,-0.002074,1.17,1.17,0,BUY
1,2018-07-24,ABBEYBDS,1.17,0.0,0.000000,1.248,1.274,1.2870,1.2948,1.257025,...,-0.1,0.0,1.367026,1.206974,-0.018377,-0.005335,1.17,1.17,0,BUY
2,2018-07-25,ABBEYBDS,1.17,0.0,0.000000,1.222,1.261,1.2805,1.2922,1.241202,...,-0.1,0.0,1.375750,1.185250,-0.024441,-0.009156,1.17,1.17,0,BUY
3,2018-07-26,ABBEYBDS,1.17,0.0,0.000000,1.196,1.248,1.2740,1.2896,1.228256,...,-0.1,0.0,1.380702,1.167298,-0.028913,-0.013107,1.17,1.17,0,BUY
4,2018-07-27,ABBEYBDS,1.17,0.0,0.000000,1.170,1.235,1.2675,1.2870,1.217664,...,-0.1,0.0,1.383008,1.151992,-0.032087,-0.016903,1.17,1.17,0,BUY


In [30]:
df.signal.unique()

array(['BUY'], dtype=object)

In [31]:
import numpy as np

df = df.sort_values(["stock_name", "date"]).copy()

df["target_return_5d"] = (
    df.groupby("stock_name")["close"]
      .transform(lambda x: (x.shift(-5) - x) / x)
)

In [32]:
df.head()

,date,stock_name,close,return,log_return,ma5,ma10,ma20,ma50,ema10,...,rsi,bb_upper,bb_lower,macd,macd_signal,target,target_next_5_days,stock_id,signal,target_return_5d
0,2018-07-23,ABBEYBDS,1.17,-0.1,-0.105361,1.274,1.287,1.2935,1.2974,1.276364,...,0.0,1.351638,1.235362,-0.010370,-0.002074,1.17,1.17,0,BUY,0.0
1,2018-07-24,ABBEYBDS,1.17,0.0,0.000000,1.248,1.274,1.2870,1.2948,1.257025,...,0.0,1.367026,1.206974,-0.018377,-0.005335,1.17,1.17,0,BUY,0.0
2,2018-07-25,ABBEYBDS,1.17,0.0,0.000000,1.222,1.261,1.2805,1.2922,1.241202,...,0.0,1.375750,1.185250,-0.024441,-0.009156,1.17,1.17,0,BUY,0.0
3,2018-07-26,ABBEYBDS,1.17,0.0,0.000000,1.196,1.248,1.2740,1.2896,1.228256,...,0.0,1.380702,1.167298,-0.028913,-0.013107,1.17,1.17,0,BUY,0.0
4,2018-07-27,ABBEYBDS,1.17,0.0,0.000000,1.170,1.235,1.2675,1.2870,1.217664,...,0.0,1.383008,1.151992,-0.032087,-0.016903,1.17,1.17,0,BUY,0.0


In [33]:
df.target_return_5d.min()

np.float64(-0.9889380530973452)

In [34]:
df.head()

,date,stock_name,close,return,log_return,ma5,ma10,ma20,ma50,ema10,...,rsi,bb_upper,bb_lower,macd,macd_signal,target,target_next_5_days,stock_id,signal,target_return_5d
0,2018-07-23,ABBEYBDS,1.17,-0.1,-0.105361,1.274,1.287,1.2935,1.2974,1.276364,...,0.0,1.351638,1.235362,-0.010370,-0.002074,1.17,1.17,0,BUY,0.0
1,2018-07-24,ABBEYBDS,1.17,0.0,0.000000,1.248,1.274,1.2870,1.2948,1.257025,...,0.0,1.367026,1.206974,-0.018377,-0.005335,1.17,1.17,0,BUY,0.0
2,2018-07-25,ABBEYBDS,1.17,0.0,0.000000,1.222,1.261,1.2805,1.2922,1.241202,...,0.0,1.375750,1.185250,-0.024441,-0.009156,1.17,1.17,0,BUY,0.0
3,2018-07-26,ABBEYBDS,1.17,0.0,0.000000,1.196,1.248,1.2740,1.2896,1.228256,...,0.0,1.380702,1.167298,-0.028913,-0.013107,1.17,1.17,0,BUY,0.0
4,2018-07-27,ABBEYBDS,1.17,0.0,0.000000,1.170,1.235,1.2675,1.2870,1.217664,...,0.0,1.383008,1.151992,-0.032087,-0.016903,1.17,1.17,0,BUY,0.0


In [35]:
df.describe() 

,date,close,return,log_return,ma5,ma10,ma20,ma50,ema10,ema20,...,drawdown,rsi,bb_upper,bb_lower,macd,macd_signal,target,target_next_5_days,stock_id,target_return_5d
count,207153,207153.000000,207153.000000,207153.000000,207153.000000,207153.000000,207153.000000,207153.000000,207153.000000,207153.000000,...,207153.000000,207153.000000,207153.000000,207153.000000,207153.000000,207153.000000,207153.000000,207153.000000,207153.000000,206448.000000
mean,2022-01-11 18:59:43.264543488,62.210149,0.002369,0.001197,61.908458,61.525790,60.777711,58.955247,61.533225,60.845285,...,-0.335383,50.230163,66.169164,55.386257,0.918847,0.872629,62.358381,62.899991,74.453698,0.010847
min,2016-08-08 00:00:00,0.200000,-0.988938,-4.504244,0.200000,0.200000,0.200500,0.200200,0.200110,0.200235,...,-0.988943,0.000000,0.204972,-58.195915,-191.452460,-167.238322,0.200000,0.200000,0.000000,-0.988938
25%,2019-08-19 00:00:00,1.260000,0.000000,0.000000,1.258000,1.254000,1.251500,1.238400,1.255121,1.251121,...,-0.505618,33.333333,1.402367,1.073704,-0.050544,-0.045240,1.260000,1.260000,39.000000,-0.031011
50%,2022-04-01 00:00:00,4.400000,0.000000,0.000000,4.362000,4.350000,4.321500,4.267400,4.351123,4.327772,...,-0.320755,50.000000,4.797198,3.826256,0.001477,0.001737,4.400000,4.400000,73.000000,0.000000
75%,2024-07-04 00:00:00,20.000000,0.000212,0.000212,20.100000,20.065000,19.965000,19.684000,20.084100,19.998718,...,-0.146667,68.000000,21.888269,17.938031,0.126178,0.122293,20.050000,20.180000,112.000000,0.037037
max,2026-06-19 00:00:00,11600.000000,89.400000,4.504244,11600.000000,11568.500000,11527.790000,11029.512000,11483.414939,11374.216734,...,0.000000,100.000000,12251.398979,11418.854212,697.157027,630.040143,11600.000000,11600.000000,145.000000,89.400000
std,NaN,331.890535,0.201519,0.040263,329.398560,326.233793,319.998941,304.739086,326.190607,320.384944,...,0.223992,28.917965,347.164238,294.265398,12.572433,11.736381,332.929883,336.913910,42.131961,0.224519


In [36]:
df.head()

,date,stock_name,close,return,log_return,ma5,ma10,ma20,ma50,ema10,...,rsi,bb_upper,bb_lower,macd,macd_signal,target,target_next_5_days,stock_id,signal,target_return_5d
0,2018-07-23,ABBEYBDS,1.17,-0.1,-0.105361,1.274,1.287,1.2935,1.2974,1.276364,...,0.0,1.351638,1.235362,-0.010370,-0.002074,1.17,1.17,0,BUY,0.0
1,2018-07-24,ABBEYBDS,1.17,0.0,0.000000,1.248,1.274,1.2870,1.2948,1.257025,...,0.0,1.367026,1.206974,-0.018377,-0.005335,1.17,1.17,0,BUY,0.0
2,2018-07-25,ABBEYBDS,1.17,0.0,0.000000,1.222,1.261,1.2805,1.2922,1.241202,...,0.0,1.375750,1.185250,-0.024441,-0.009156,1.17,1.17,0,BUY,0.0
3,2018-07-26,ABBEYBDS,1.17,0.0,0.000000,1.196,1.248,1.2740,1.2896,1.228256,...,0.0,1.380702,1.167298,-0.028913,-0.013107,1.17,1.17,0,BUY,0.0
4,2018-07-27,ABBEYBDS,1.17,0.0,0.000000,1.170,1.235,1.2675,1.2870,1.217664,...,0.0,1.383008,1.151992,-0.032087,-0.016903,1.17,1.17,0,BUY,0.0


In [37]:
df.head()

,date,stock_name,close,return,log_return,ma5,ma10,ma20,ma50,ema10,...,rsi,bb_upper,bb_lower,macd,macd_signal,target,target_next_5_days,stock_id,signal,target_return_5d
0,2018-07-23,ABBEYBDS,1.17,-0.1,-0.105361,1.274,1.287,1.2935,1.2974,1.276364,...,0.0,1.351638,1.235362,-0.010370,-0.002074,1.17,1.17,0,BUY,0.0
1,2018-07-24,ABBEYBDS,1.17,0.0,0.000000,1.248,1.274,1.2870,1.2948,1.257025,...,0.0,1.367026,1.206974,-0.018377,-0.005335,1.17,1.17,0,BUY,0.0
2,2018-07-25,ABBEYBDS,1.17,0.0,0.000000,1.222,1.261,1.2805,1.2922,1.241202,...,0.0,1.375750,1.185250,-0.024441,-0.009156,1.17,1.17,0,BUY,0.0
3,2018-07-26,ABBEYBDS,1.17,0.0,0.000000,1.196,1.248,1.2740,1.2896,1.228256,...,0.0,1.380702,1.167298,-0.028913,-0.013107,1.17,1.17,0,BUY,0.0
4,2018-07-27,ABBEYBDS,1.17,0.0,0.000000,1.170,1.235,1.2675,1.2870,1.217664,...,0.0,1.383008,1.151992,-0.032087,-0.016903,1.17,1.17,0,BUY,0.0


In [38]:
df.tail()

,date,stock_name,close,return,log_return,ma5,ma10,ma20,ma50,ema10,...,rsi,bb_upper,bb_lower,macd,macd_signal,target,target_next_5_days,stock_id,signal,target_return_5d
207148,2026-06-15,ZICHIS,30.00,-0.032258,-0.032790,30.630,31.071,31.1560,23.9300,30.870974,...,37.537388,35.183591,27.128409,1.516246,2.294539,31.00,23.40,145,BUY,NaN
207149,2026-06-16,ZICHIS,31.00,0.033333,0.032790,30.660,30.771,31.0715,24.2468,30.894433,...,43.277524,35.033967,27.109033,1.413888,2.118409,31.00,24.67,145,BUY,NaN
207150,2026-06-17,ZICHIS,31.00,0.000000,0.000000,30.730,30.633,31.1500,24.5938,30.913627,...,50.058685,35.037031,27.262969,1.317577,1.958243,28.12,24.90,145,BUY,NaN
207151,2026-06-18,ZICHIS,28.12,-0.092903,-0.097506,30.224,30.525,31.2315,24.8968,30.405695,...,45.036959,34.758844,27.704156,0.997747,1.766143,26.00,24.90,145,BUY,NaN
207152,2026-06-19,ZICHIS,26.00,-0.075391,-0.078385,29.224,30.165,31.0750,25.1340,29.604659,...,30.620155,35.218802,26.931198,0.566947,1.526304,23.40,24.95,145,BUY,NaN


In [39]:
df.info(99)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 207153 entries, 0 to 207152
Data columns (total 27 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   date                207153 non-null  datetime64[ns]
 1   stock_name          207153 non-null  object        
 2   close               207153 non-null  float64       
 3   return              207153 non-null  float64       
 4   log_return          207153 non-null  float64       
 5   ma5                 207153 non-null  float64       
 6   ma10                207153 non-null  float64       
 7   ma20                207153 non-null  float64       
 8   ma50                207153 non-null  float64       
 9   ema10               207153 non-null  float64       
 10  ema20               207153 non-null  float64       
 11  volatility10        207153 non-null  float64       
 12  momentum10          207153 non-null  float64       
 13  roc10               207153 no